# **Trabalho Final POO - Análise de Sentimentos**
## **Disciplina: Programação Orientada a Objetos**
## **Professor: Dirson**
## **Turma: D01**

## **Alunos:**
**- Gustavo Rodrigues Ribeiro / RA:202003570** \
**- Breno Machado Barros / RA:202014607** \

# **Domínio do Negócio: E-commerce**
### Para este projeto, o domínio de negócio escolhido será o de e-commerce com foco na análise de sentimentos em avaliações de produtos. As avaliações são obtidas de um dataset público de avaliações de produtos eletrônicos, como o Amazon Product Reviews Dataset, que oferece avaliações reais em várias categorias de produtos.

## Iniciando o PySpark

Esta célula de código instala o Spark no ambiente de execução Colab. Aqui está uma explicação passo a passo:

1. **`!apt-get install openjdk-11-jdk-headless -qq > /dev/null`**: este comando instala o OpenJDK 11 (versão headless, sem interface gráfica), que é um requisito para o Spark. O `-qq` suprime a saída e o `> /dev/null` redireciona a saída para o nada, tornando o processo mais silencioso.

2. **`!wget -q https://dlcdn.apache.org/spark/spark-3.5.2/spark-3.5.3-bin-hadoop3.tgz`**: Este comando baixa o arquivo compactado do Spark 3.5.2 (construído para o Hadoop 3) do site oficial do Apache Spark. O `-q` suprime a saída de download.

3. **`!tar xf spark-3.5.3-bin-hadoop3.tgz`**: Este comando extrai o arquivo compactado baixado do Spark, criando um diretório chamado `spark-3.5.3-bin-hadoop3`.

4. **`!pip -q install findspark`**: Este comando instala a biblioteca `findspark` usando `pip`. Findspark é uma biblioteca Python que torna mais fácil configurar o Spark em um ambiente Python, principalmente no Colab. Ela define as variáveis de ambiente necessárias para que o Spark funcione corretamente.

Após executar essas linhas, você terá o Spark instalado e pronto para ser usado em seu notebook Colab.

In [ ]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://dlcdn.apache.org/spark/spark-3.5.3/spark-3.5.3-bin-hadoop3.tgz
!tar xf spark-3.5.3-bin-hadoop3.tgz
!pip -q install findspark

Defina as variáveis de ambiente do Spark:

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.3-bin-hadoop3"

O código a seguir garante que o Spark seja configurado corretamente e esteja pronto para uso em seu ambiente Python.

* **`findspark.init()`**: executa a função `init()` do módulo `findspark`. Esta função:
    * Localiza a instalação do Spark em seu sistema.
    * Configura as variáveis de ambiente necessárias para que o Python possa interagir com o Spark. Isso permite que o driver Python (seu código Python) se comunique com o executor Spark (o código que realmente processa os dados).


In [ ]:
import findspark
findspark.init()

Depois de executar a célula anterior, você poderá importar e usar as bibliotecas Spark como `pyspark.sql.SparkSession` para criar uma sessão Spark e começar a trabalhar com dados.

**OBS: Vale lembra que esse é um código para a criação de um modelo de treinamento e teste de IA para análise de sentimento através de um dataset de avaliações de produtos, com cerca de 6000000 de reviews. Logo, é importante entender que apenas o Colab (versão gratuita) não possui recursos computacionais (GPU e RAM) suficientes para executar o modelo por completo. Assim, recomendamos a utilização da máquina local com cerca de 64gb de RAM ou uma máquina virtual como a N-highmem-64gb no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook. Assim, o código irá executar sem erros de memória ou GPU.**

**OBS 2: Caso utilize uma máquina virtual como a N-highmem-64gb no Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, não serão necessários os passos acima, apenas continue daqui.**

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.ml.feature import *
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.storagelevel import StorageLevel

# Caso necessário utilize spark.stop() para encerrar a sessão atual e reiniciar a SparkSession com as configurações abaixo
# spark.stop()

spark = SparkSession.builder.appName('Trabalho Final Gold').config("spark.driver.memory", "64g").config("spark.executor.memory", "64g").config("spark.executor.cores", "8").master("local[*]").getOrCreate()
print("Versão do Spark:", spark.version)

Versão do Spark: 3.5.1


## **Arquitetura Medallion: GOLD**
## **Feature Engineering e Modelagem de IA**

Aqui estaremos sincronizando nossa conta no Drive ao ambiente Colab, para que os arquivos em nuvem sejam gerenciados (lidos e escritos) e manipulados diretamente no Drive.

**OBS: Caso esteja utilizando o Dataproc do Google Cloud Console, com o ambiente virtual Jupyter Notebook, você podera utilizar o Data Lake Google Cloud Storage (GCS) que está conectado a sua conta, não necessitando desse processo de sincronização com o Drive.**

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")

# !ls /content/drive

In [ ]:
# -------------------------------------------------
# Camada Gold: Dados Agregados e Treinamento da IA
# -------------------------------------------------

# Carregar dados da camada Silver no GCS ou Drive
dataset_path = "gs://pdm-gustavorr-2024-2/Silver/reviews_silver"  # Mude para o diretório desejado

# Ler os dados diretamente da camada Bronze no GCS ou Drive em .parquet
reviews_df_gold = spark.read.parquet(dataset_path)

#----------------------------------------------------------------------------------

# Conta o número de linhas no DataFrame
num_linhas = reviews_df_gold.count()
# Imprime o resultado
print()
print(f"A tabela original contém {num_linhas} registros.")
print()

# ---------------------------------------------------------------------------------

# Dividir o conjunto de dados em treino (80%) e teste (20%)
train_df, test_df = reviews_df_gold.randomSplit([0.8, 0.2], seed=42)

# Persistir os DataFrames
train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df.persist(StorageLevel.MEMORY_AND_DISK)

#### -------------------------------------- Feature Engineering ----------------------------------- ####

# Tokenizar o texto da avaliação
tokenizer = Tokenizer(inputCol="reviewText", outputCol="words")

# Remover stop words
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filteredWords")

# Vectorização das palavras com CountVectorizer
vectorizer = CountVectorizer(inputCol="filteredWords", outputCol="rawFeatures")

# Aplicar TF-IDF
idf = IDF(inputCol="rawFeatures", outputCol="features")

#### ------------------------- Balanceamento de Classes (Class Weights) ----------------------------- ####

# Contar a quantidade de amostras por classe e total
class_distribution = reviews_df_gold.groupBy("sentimentOverall").count()

# Calcular o total de registros
total_count = class_distribution.agg(sum("count")).collect()[0][0]

# Criar um DataFrame com os pesos das classes
class_weights_df = class_distribution.withColumn(
    "classWeight", lit(total_count) / col("count")
)

class_weights = {row['sentimentOverall']: row["classWeight"] for row in class_weights_df.collect()}

def add_class_weights(df, class_weights):
    weight_udf = udf(lambda sentiment: class_weights[sentiment], FloatType())
    return df.withColumn("classWeight", weight_udf(col("sentimentOverall")))

# Aplicar o Class Weights no DataFrame de teste
train_df_weighted = add_class_weights(train_df, class_weights)

#### ------------------------- Balanceamento de Classes (Undersampling) ----------------------------- ####

# Contar a quantidade de amostras por classe
class_distribution_test = test_df.groupBy("sentimentOverall").count()

# Identificar a classe minoritária
minority_class_count_test = class_distribution_test.orderBy("count").first()["count"]

# Função para realizar o undersampling nas classes majoritárias
def undersample(df: DataFrame, label_col: str, minority_class_count: int, class_distribution: DataFrame) -> DataFrame:
    sampled_dfs = []

    # Iterar sobre cada classe
    for row in class_distribution.collect():
        label = row["sentimentOverall"]
        count = row["count"]

        # Se a classe for majoritária, aplicar undersampling
        if count > minority_class_count:
            sampled_df = df.filter(col(label_col) == label).sample(fraction=minority_class_count / count, seed=42)
        else:
            sampled_df = df.filter(col(label_col) == label)

        sampled_dfs.append(sampled_df)

    # Combinar todas as classes balanceadas
    balanced_df = sampled_dfs[0]
    for df_part in sampled_dfs[1:]:
        balanced_df = balanced_df.union(df_part)

    return balanced_df

# Aplicar o undersampling no DataFrame de teste
distributed_test_df = undersample(test_df, "sentimentOverall", minority_class_count_test, class_distribution_test)

#### -------------------------------------- Modelagem de IA ---------------------------------------- ####

# Definir o modelo de regressão logística para multiclasse
lr = LogisticRegression(labelCol="sentimentOverall", featuresCol="features", maxIter=10, family="multinomial", weightCol="classWeight")

# Construir o pipeline
pipeline = Pipeline(stages=[tokenizer, stopwords_remover, vectorizer, idf, lr])

#### ----------------------------------------------------------------------------------------------- ####

print("#### ---------------- Tabela Teste Distribuida ---------------- ####")

# Verificar a nova distribuição de classes
distributed_test_df.groupBy("sentimentOverall").count().show()

# Conta o número de linhas no DataFrame
num_linhas = distributed_test_df.count()
# Imprime o resultado
print(f"A tabela test contém {num_linhas} registros.")
print()

#### ---------------------------------- Treinamento do Modelo -------------------------------------- ####

# Treinar o modelo
spark_model = pipeline.fit(train_df_weighted)

print()
print("Modelo Treinado")
print()


A tabela original contém 5802834 registros.



#### ---------------- Tabela Teste Distribuida ---------------- ####
+----------------+-----+
|sentimentOverall|count|
+----------------+-----+
|               1|84937|
|               2|84987|
|               0|85301|
+----------------+-----+



A tabela test contém 255225 registros.



24/12/17 19:43:01 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
24/12/17 19:43:42 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
24/12/17 19:43:43 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:44:27 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:44:28 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:45:17 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:45:17 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:45:20 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:45:20 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:45:22 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:45:23 WARN DAGScheduler: Broadcasting large task binary with size 7.1 MiB
24/12/17 19:45:25 WARN DAGScheduler: Broadcasting larg


Modelo Treinado



In [ ]:
#### --------------------------------------- Avaliação do Modelo ----------------------------------------- ####

# Previsão no conjunto de dados de teste
predictions = spark_model.transform(distributed_test_df)

# Avaliar acurácia
accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="sentimentOverall", predictionCol="prediction", metricName="accuracy")
accuracy = accuracy_evaluator.evaluate(predictions)

# Avaliar precisão média
precision_evaluator = MulticlassClassificationEvaluator(labelCol="sentimentOverall", predictionCol="prediction", metricName="weightedPrecision")
precision = precision_evaluator.evaluate(predictions)

# Avaliar revocação
recall_evaluator = MulticlassClassificationEvaluator(labelCol="sentimentOverall", predictionCol="prediction", metricName="weightedRecall")
recall = recall_evaluator.evaluate(predictions)

# Avaliar F1-Score
f1_evaluator = MulticlassClassificationEvaluator(labelCol="sentimentOverall", predictionCol="prediction", metricName="f1")
f1_score = f1_evaluator.evaluate(predictions)

# Imprimir os resultados
print()
print(f"Acurácia: {accuracy:.2f}")
print(f"Precisão Média: {precision:.2f}")
print(f"Revocação Média: {recall:.2f}")
print(f"F1-Score: {f1_score:.2f}")

24/12/17 19:45:48 WARN DAGScheduler: Broadcasting large task binary with size 13.2 MiB
24/12/17 19:45:52 WARN DAGScheduler: Broadcasting large task binary with size 13.2 MiB
24/12/17 19:45:56 WARN DAGScheduler: Broadcasting large task binary with size 13.2 MiB
24/12/17 19:46:01 WARN DAGScheduler: Broadcasting large task binary with size 13.2 MiB



Acurácia: 0.68
Precisão Média: 0.68
Revocação Média: 0.68
F1-Score: 0.68


In [ ]:
# Salva o Modelo na camada Gold
# obs: Altere o caminho do arquivo para sua preferência, desde que esteja em seu Drive ou GCS
spark_model.write().overwrite().save("gs://pdm-gustavorr-2024-2/Gold/sentiment_classification_model4_gold")

print()
print("Modelo Salvo")
print()

24/12/17 19:46:21 WARN TaskSetManager: Stage 71 contains a task of very large size (5255 KiB). The maximum recommended task size is 1000 KiB.
24/12/17 19:46:28 WARN TaskSetManager: Stage 75 contains a task of very large size (4187 KiB). The maximum recommended task size is 1000 KiB.
24/12/17 19:46:37 WARN TaskSetManager: Stage 79 contains a task of very large size (6278 KiB). The maximum recommended task size is 1000 KiB.



Modelo Salvo

